In [2]:
start_year = "2026"
end_year = "2027"
start_month = "06"

source_type = "Dataverse"

# Multiple raw source folders
source_object_names = "client,course_of_licbt,iar_dst,intake,outbound_referral,outcomes, practitioner,practitioner_preference, privacy_preferences, questionnaire_material, questionnaire_response, questionnaire_session, service_contract"

create_raw = "true"
create_processed = "true"
create_logs = "true"

create_object_folder_in_processed = "false"
create_object_folder_in_logs = "false"

In [3]:
# ============================================================
# Generic notebook to create yearly/monthly/weekly Lakehouse folder structure
#
# Raw path:
# /lakehouse/default/Files/raw/Dataverse/YYYY/MM/week01/sourcefoldername
#
# Processed path:
# /lakehouse/default/Files/processed/Dataverse/YYYY/MM/week01
#
# Logs path:
# /lakehouse/default/Files/logs/Dataverse/YYYY/MM
# ============================================================

import notebookutils


# ============================================================
# Helper functions
# ============================================================

def to_bool(value):
    """
    Convert string parameter to boolean.
    """
    return str(value).strip().lower() in ["true", "1", "yes", "y"]


def validate_required_params():
    """
    Validate required notebook parameters.
    """
    if not str(start_year).strip():
        raise Exception("start_year is mandatory.")

    if not str(end_year).strip():
        raise Exception("end_year is mandatory.")

    if not str(start_month).strip():
        raise Exception("start_month is mandatory.")

    if not str(source_type).strip():
        raise Exception("source_type is mandatory.")

    if not str(source_object_names).strip():
        raise Exception("source_object_names is mandatory.")

    if not str(start_year).isdigit() or len(str(start_year).strip()) != 4:
        raise Exception("start_year must be in YYYY format. Example: 2026")

    if not str(end_year).isdigit() or len(str(end_year).strip()) != 4:
        raise Exception("end_year must be in YYYY format. Example: 2027")

    if not str(start_month).isdigit():
        raise Exception("start_month must be numeric. Example: 06")

    if int(start_year) > int(end_year):
        raise Exception("start_year cannot be greater than end_year.")

    if int(start_month) < 1 or int(start_month) > 12:
        raise Exception("start_month must be between 01 and 12.")


def create_folder(path):
    """
    Create folder in Lakehouse Files section.
    """
    notebookutils.fs.mkdirs(path)
    print(f"Created/verified folder: {path}")


def get_months_for_year(current_year, start_year_int, start_month_int):
    """
    For start year, start from given start_month.
    For next years, start from January.
    """
    if current_year == start_year_int:
        return [f"{m:02d}" for m in range(start_month_int, 13)]

    return [f"{m:02d}" for m in range(1, 13)]


def parse_source_objects(source_object_names):
    """
    Convert comma-separated source object names into a clean list.
    Example:
    practitioner,bookableresource,role
    """
    return [
        item.strip()
        for item in str(source_object_names).split(",")
        if item.strip()
    ]


# ============================================================
# Main execution
# ============================================================

validate_required_params()

start_year_int = int(start_year)
end_year_int = int(end_year)
start_month_int = int(start_month)

source_type_clean = str(source_type).strip()
source_object_list = parse_source_objects(source_object_names)

create_raw_flag = to_bool(create_raw)
create_processed_flag = to_bool(create_processed)
create_logs_flag = to_bool(create_logs)

create_object_folder_in_processed_flag = to_bool(create_object_folder_in_processed)
create_object_folder_in_logs_flag = to_bool(create_object_folder_in_logs)

weeks = [f"week{w:02d}" for w in range(1, 6)]

created_folders = []

for year in range(start_year_int, end_year_int + 1):
    year_str = str(year)

    months = get_months_for_year(
        current_year=year,
        start_year_int=start_year_int,
        start_month_int=start_month_int
    )

    for month in months:

        # ========================================================
        # Logs folder - monthly level only
        # Example:
        # /lakehouse/default/Files/logs/Dataverse/2026/06
        # ========================================================
        if create_logs_flag:
            if create_object_folder_in_logs_flag:
                for source_object_name in source_object_list:
                    logs_path = (
                        f"/lakehouse/default/Files/logs/{source_type_clean}/"
                        f"{year_str}/{month}/{source_object_name}"
                    )
                    create_folder(logs_path)
                    created_folders.append(logs_path)
            else:
                logs_path = (
                    f"/lakehouse/default/Files/logs/{source_type_clean}/"
                    f"{year_str}/{month}"
                )
                create_folder(logs_path)
                created_folders.append(logs_path)

        for week in weeks:

            # ====================================================
            # Raw folders - weekly + source object level
            # Example:
            # /lakehouse/default/Files/raw/Dataverse/2026/06/week01/practitioner
            # /lakehouse/default/Files/raw/Dataverse/2026/06/week01/bookableresource
            # ====================================================
            if create_raw_flag:
                for source_object_name in source_object_list:
                    raw_path = (
                        f"/lakehouse/default/Files/raw/{source_type_clean}/"
                        f"{year_str}/{month}/{week}/{source_object_name}"
                    )
                    create_folder(raw_path)
                    created_folders.append(raw_path)

            # ====================================================
            # Processed folder - weekly level only by default
            # Example:
            # /lakehouse/default/Files/processed/Dataverse/2026/06/week01
            # ====================================================
            if create_processed_flag:
                if create_object_folder_in_processed_flag:
                    for source_object_name in source_object_list:
                        processed_path = (
                            f"/lakehouse/default/Files/processed/{source_type_clean}/"
                            f"{year_str}/{month}/{week}/{source_object_name}"
                        )
                        create_folder(processed_path)
                        created_folders.append(processed_path)
                else:
                    processed_path = (
                        f"/lakehouse/default/Files/processed/{source_type_clean}/"
                        f"{year_str}/{month}/{week}"
                    )
                    create_folder(processed_path)
                    created_folders.append(processed_path)


print("Folder structure creation completed successfully.")
print(f"Start year       : {start_year}")
print(f"End year         : {end_year}")
print(f"Start month      : {str(start_month).zfill(2)}")
print(f"Source type      : {source_type_clean}")
print(f"Source objects   : {source_object_list}")
print(f"Total folders    : {len(created_folders)}")

Created/verified folder: /lakehouse/default/Files/logs/Dataverse/2026/06
Created/verified folder: /lakehouse/default/Files/raw/Dataverse/2026/06/week01/client
Created/verified folder: /lakehouse/default/Files/raw/Dataverse/2026/06/week01/course_of_licbt
Created/verified folder: /lakehouse/default/Files/raw/Dataverse/2026/06/week01/iar_dst
Created/verified folder: /lakehouse/default/Files/raw/Dataverse/2026/06/week01/intake
Created/verified folder: /lakehouse/default/Files/raw/Dataverse/2026/06/week01/outbound_referral
Created/verified folder: /lakehouse/default/Files/raw/Dataverse/2026/06/week01/outcomes
Created/verified folder: /lakehouse/default/Files/raw/Dataverse/2026/06/week01/practitioner
Created/verified folder: /lakehouse/default/Files/raw/Dataverse/2026/06/week01/practitioner_preference
Created/verified folder: /lakehouse/default/Files/raw/Dataverse/2026/06/week01/privacy_preferences
Created/verified folder: /lakehouse/default/Files/raw/Dataverse/2026/06/week01/questionnaire_m